# Lung Segmentation with U-Net (PyTorch)
Chest X-ray lung segmentation on the Kaggle dataset `newra008/lung-mask-image-dataset` (binary masks, 256x256 input, 4-level U-Net, BCE + Dice loss).

**Run order:** top to bottom. Turn on a GPU first (Runtime -> Change runtime type -> T4 GPU). Training takes about 1 hour (8 epochs, ~7 min each).

## 1. Setup

In [ ]:
import os, time
import cv2
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import matplotlib.pyplot as plt

torch.manual_seed(42); np.random.seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Torch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())
print("device:", device)

## 2. Download the dataset from Kaggle
Upload your own `kaggle.json` (Kaggle -> Settings -> API -> Create New Token). Never commit this file to GitHub.

In [ ]:
from google.colab import files
files.upload()   # choose kaggle.json

In [ ]:
os.makedirs("/root/.kaggle", exist_ok=True)
!mv kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json

In [ ]:
!kaggle datasets download -d newra008/lung-mask-image-dataset
!unzip -q lung-mask-image-dataset.zip -d /content/lung_data

In [ ]:
for dp, dn, fn in os.walk("/content/lung_data"):
    if fn:
        print(dp, len(fn), fn[:2])

## 3. Dataset
Folders: `ChestXray/{train,val,test}/{image,mask}`. Images are `cxrimage_N.png`, masks are `cxrmask_N.jpeg` (same number N). Masks are JPEG, so they are thresholded at 127 instead of divided by 255.

In [ ]:
class LungDataset(Dataset):
    def __init__(self, split_dir, img_size=(256, 256)):
        self.image_dir = os.path.join(split_dir, "image")
        self.mask_dir  = os.path.join(split_dir, "mask")
        self.img_size  = img_size
        self.filenames = sorted(os.listdir(self.image_dir))

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        name = self.filenames[idx]
        number = name.replace("cxrimage_", "").split(".")[0]
        mask_name = f"cxrmask_{number}.jpeg"

        image_path = os.path.join(self.image_dir, name)
        mask_path  = os.path.join(self.mask_dir, mask_name)

        image = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
        mask  = cv2.imread(mask_path, cv2.IMREAD_GRAYSCALE)
        if image is None:
            raise FileNotFoundError(f"Could not read image: {image_path}")
        if mask is None:
            raise FileNotFoundError(f"Could not read mask: {mask_path}")

        image = cv2.resize(image, self.img_size, interpolation=cv2.INTER_LINEAR)
        mask  = cv2.resize(mask,  self.img_size, interpolation=cv2.INTER_NEAREST)

        image = image.astype(np.float32) / 255.0
        mask  = (mask > 127).astype(np.float32)   # threshold: JPEG masks are noisy

        image = torch.from_numpy(image).unsqueeze(0)   # (1, H, W)
        mask  = torch.from_numpy(mask).unsqueeze(0)    # (1, H, W)
        return image, mask

In [ ]:
base = "/content/lung_data/ChestXray"
train_ds = LungDataset(f"{base}/train")
val_ds   = LungDataset(f"{base}/val")
test_ds  = LungDataset(f"{base}/test")

img, mask = train_ds[0]
print(len(train_ds), len(val_ds), len(test_ds))
print(img.shape, mask.shape, mask.unique())

fig, ax = plt.subplots(1, 3, figsize=(10, 4))
ax[0].imshow(img[0], cmap="gray");  ax[0].set_title("image")
ax[1].imshow(mask[0], cmap="gray"); ax[1].set_title("mask")
ax[2].imshow(img[0], cmap="gray"); ax[2].imshow(mask[0], alpha=0.4, cmap="Reds"); ax[2].set_title("overlay")
for a in ax: a.axis("off")
plt.show()

## 4. DataLoaders

In [ ]:
train_loader = DataLoader(train_ds, batch_size=16, shuffle=True,  num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

images, masks = next(iter(train_loader))
print(images.shape, masks.shape)       # [16, 1, 256, 256] each
print(len(train_loader), len(val_loader), len(test_loader))   # 1059 133 133

## 5. Model: 4-level U-Net

In [ ]:
class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)


class UNet(nn.Module):
    def __init__(self, in_ch=1, out_ch=1, base=32):
        super().__init__()
        self.pool = nn.MaxPool2d(2)

        # Encoder
        self.enc1 = DoubleConv(in_ch,  base)
        self.enc2 = DoubleConv(base,   base*2)
        self.enc3 = DoubleConv(base*2, base*4)
        self.enc4 = DoubleConv(base*4, base*8)

        # Bottom
        self.bottleneck = DoubleConv(base*8, base*16)

        # Decoder
        self.up4  = nn.ConvTranspose2d(base*16, base*8, kernel_size=2, stride=2)
        self.dec4 = DoubleConv(base*16, base*8)
        self.up3  = nn.ConvTranspose2d(base*8, base*4, kernel_size=2, stride=2)
        self.dec3 = DoubleConv(base*8, base*4)
        self.up2  = nn.ConvTranspose2d(base*4, base*2, kernel_size=2, stride=2)
        self.dec2 = DoubleConv(base*4, base*2)
        self.up1  = nn.ConvTranspose2d(base*2, base, kernel_size=2, stride=2)
        self.dec1 = DoubleConv(base*2, base)

        # 1x1 conv: base channels -> 1 channel (lung or not). Outputs logits (no sigmoid).
        self.head = nn.Conv2d(base, out_ch, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        e4 = self.enc4(self.pool(e3))
        b  = self.bottleneck(self.pool(e4))

        d4 = self.dec4(torch.cat([self.up4(b),  e4], dim=1))
        d3 = self.dec3(torch.cat([self.up3(d4), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)

In [ ]:
model = UNet()
x = torch.randn(2, 1, 256, 256)
print(model(x).shape)    # [2, 1, 256, 256]
print(sum(p.numel() for p in model.parameters()) / 1e6, "million parameters")   # ~7.77

## 6. Loss and metrics

In [ ]:
class BCEDiceLoss(nn.Module):
    def __init__(self, smooth=1e-6):
        super().__init__()
        self.smooth = smooth
        self.bce = nn.BCEWithLogitsLoss()

    def forward(self, logits, targets):
        bce_loss = self.bce(logits, targets)

        probs   = torch.sigmoid(logits).view(logits.size(0), -1)
        targets = targets.view(targets.size(0), -1)

        intersection = (probs * targets).sum(dim=1)
        dice = (2 * intersection + self.smooth) / (probs.sum(dim=1) + targets.sum(dim=1) + self.smooth)
        return bce_loss + (1 - dice.mean())


@torch.no_grad()
def dice_iou(logits, targets, threshold=0.5, smooth=1e-6):
    preds   = (torch.sigmoid(logits) > threshold).float()
    preds   = preds.view(preds.size(0), -1)
    targets = targets.view(targets.size(0), -1)

    intersection = (preds * targets).sum(dim=1)
    union = preds.sum(dim=1) + targets.sum(dim=1) - intersection

    dice = (2 * intersection + smooth) / (preds.sum(dim=1) + targets.sum(dim=1) + smooth)
    iou  = (intersection + smooth) / (union + smooth)
    return dice.mean().item(), iou.mean().item()

In [ ]:
# Sanity check on an untrained model: loss ~1.4, Dice ~0.4 is expected
model = UNet().to(device)
criterion = BCEDiceLoss()

images, masks = next(iter(train_loader))
images, masks = images.to(device), masks.to(device)
logits = model(images)
print("loss:", criterion(logits, masks).item())
print("dice, iou:", dice_iou(logits, masks))

## 7. Train / evaluate function

In [ ]:
def run_epoch(model, loader, criterion, optimizer=None):
    """Train if an optimizer is given, otherwise just evaluate."""
    training = optimizer is not None
    model.train() if training else model.eval()

    total_loss, total_dice, total_iou, n = 0.0, 0.0, 0.0, 0

    with torch.set_grad_enabled(training):
        for images, masks in loader:
            images, masks = images.to(device), masks.to(device)

            logits = model(images)
            loss = criterion(logits, masks)

            if training:
                optimizer.zero_grad()   # clear old gradients
                loss.backward()         # compute new gradients
                optimizer.step()        # update the weights

            bs = images.size(0)
            d, i = dice_iou(logits, masks)
            total_loss += loss.item() * bs
            total_dice += d * bs
            total_iou  += i * bs
            n += bs

    return total_loss / n, total_dice / n, total_iou / n

## 8. Google Drive (so checkpoints survive a Colab disconnect)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
CKPT = "/content/drive/MyDrive/best_unet_full.pth"
HIST = "/content/drive/MyDrive/history.pt"

## 9. Training (8 epochs, about 1 hour)
The best model (by validation Dice) is saved to Drive after every improvement. The history is also saved each epoch.

In [ ]:
print("device:", device)   # must say cuda

model = UNet().to(device)
criterion = BCEDiceLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", factor=0.5, patience=2)

EPOCHS = 8
best_dice = 0.0
history = {"train_loss": [], "val_loss": [], "train_dice": [], "val_dice": []}

for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    tr_loss, tr_dice, tr_iou = run_epoch(model, train_loader, criterion, optimizer)
    va_loss, va_dice, va_iou = run_epoch(model, val_loader, criterion)
    scheduler.step(va_dice)

    history["train_loss"].append(tr_loss); history["val_loss"].append(va_loss)
    history["train_dice"].append(tr_dice); history["val_dice"].append(va_dice)
    torch.save(history, HIST)

    saved = ""
    if va_dice > best_dice:
        best_dice = va_dice
        torch.save(model.state_dict(), CKPT)
        saved = "  <- saved"

    lr = optimizer.param_groups[0]["lr"]
    print(f"Epoch {epoch:02d} | train loss {tr_loss:.4f} dice {tr_dice:.4f} | "
          f"val loss {va_loss:.4f} dice {va_dice:.4f} iou {va_iou:.4f} | "
          f"lr {lr:.1e} | {time.time()-t0:.0f}s{saved}")

In [ ]:
# Training curves
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].plot(history["train_loss"], label="train"); ax[0].plot(history["val_loss"], label="val")
ax[0].set_title("Loss"); ax[0].set_xlabel("epoch"); ax[0].legend()
ax[1].plot(history["train_dice"], label="train"); ax[1].plot(history["val_dice"], label="val")
ax[1].set_title("Dice"); ax[1].set_xlabel("epoch"); ax[1].legend()
plt.show()

## 10. Test score (best checkpoint)

In [ ]:
criterion = BCEDiceLoss()
model = UNet().to(device)
model.load_state_dict(torch.load(CKPT, map_location=device, weights_only=True))

te_loss, te_dice, te_iou = run_epoch(model, test_loader, criterion)
print(f"TEST  loss {te_loss:.4f} | Dice {te_dice:.4f} | IoU {te_iou:.4f}")

## 11. Example predictions

In [ ]:
model.eval()
images, masks = next(iter(test_loader))
with torch.no_grad():
    preds = (torch.sigmoid(model(images.to(device))) > 0.5).float().cpu()

fig, ax = plt.subplots(4, 3, figsize=(9, 12))
for r in range(4):
    ax[r, 0].imshow(images[r, 0], cmap="gray"); ax[r, 0].set_title("image")
    ax[r, 1].imshow(masks[r, 0],  cmap="gray"); ax[r, 1].set_title("true mask")
    ax[r, 2].imshow(preds[r, 0],  cmap="gray"); ax[r, 2].set_title("predicted")
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.show()

## 12. Failure analysis: worst test images, old vs new model
Per-image Dice on the whole test set. The numbers like `#441` are **positions in the sorted file list** (test loader does not shuffle), so the cell also prints the real filenames.

The "old" model is the 1-epoch baseline. Its per-image scores were saved earlier to `baseline_scores.pt` on Drive. If that file does not exist, the comparison is skipped.

In [ ]:
model.eval()
new_scores, all_imgs, all_masks, all_preds = [], [], [], []

with torch.no_grad():
    for images, masks in test_loader:
        preds = (torch.sigmoid(model(images.to(device))) > 0.5).float().cpu()
        p = preds.view(preds.size(0), -1)
        t = masks.view(masks.size(0), -1)
        inter = (p * t).sum(dim=1)
        dice = (2 * inter + 1e-6) / (p.sum(dim=1) + t.sum(dim=1) + 1e-6)
        new_scores.extend(dice.tolist())
        all_imgs.append(images); all_masks.append(masks); all_preds.append(preds)

all_imgs, all_masks, all_preds = torch.cat(all_imgs), torch.cat(all_masks), torch.cat(all_preds)
scores = torch.tensor(new_scores)
torch.save(scores, "/content/drive/MyDrive/full_scores.pt")

print("mean Dice:", round(scores.mean().item(), 4))
print("min Dice :", round(scores.min().item(), 4))
print("images with Dice < 0.9:", (scores < 0.9).sum().item(), "of", len(scores))

# Compare with the 1-epoch baseline, if its scores are on Drive
OLD = "/content/drive/MyDrive/baseline_scores.pt"
if os.path.exists(OLD):
    old_scores = torch.load(OLD)
    print()
    print(f"{'':22}{'old':>8}{'new':>8}")
    print(f"{'mean Dice':22}{old_scores.mean():8.4f}{scores.mean():8.4f}")
    print(f"{'min Dice':22}{old_scores.min():8.4f}{scores.min():8.4f}")
    print(f"{'images with Dice<0.9':22}{(old_scores<0.9).sum().item():8d}{(scores<0.9).sum().item():8d}")

print()
worst = scores.argsort()[:4]
for i in worst.tolist():
    print(i, test_ds.filenames[i], round(scores[i].item(), 3))

fig, ax = plt.subplots(4, 3, figsize=(9, 12))
for r, i in enumerate(worst):
    ax[r, 0].imshow(all_imgs[i, 0],  cmap="gray"); ax[r, 0].set_title(f"image #{i.item()}")
    ax[r, 1].imshow(all_masks[i, 0], cmap="gray"); ax[r, 1].set_title("true mask")
    ax[r, 2].imshow(all_preds[i, 0], cmap="gray"); ax[r, 2].set_title(f"pred (Dice {scores[i]:.3f})")
    for a in ax[r]: a.axis("off")
plt.tight_layout(); plt.show()

## Results (from my run)

| Model | Val Dice | Test Dice | Test IoU | Test images with Dice < 0.9 |
|---|---|---|---|---|
| Baseline: 1 epoch | 0.959 | 0.9850 | 0.9711 | 12 of 2116 |
| 8 epochs | 0.977 | 0.9931 | 0.9864 | 4 of 2116 |

**Notes**
- Test Dice is higher than val and train Dice, which suggests the provided test split is easier than val, or overlaps with the training data. Treat the test score as "Dice on the dataset's provided test split", not real-world performance.
- Val Dice was still rising at epoch 8 and the learning rate never dropped, so more epochs would likely help.
- Weakest cases: dark/low-contrast X-rays and small, rotated chests (hypotheses to test with augmentation, CLAHE, higher resolution, and connected-component post-processing).